In [ ]:
!git clone https://github.com/Spenz7/NanoGPT-Math
%cd NanoGPT-Math

Cloning into 'NanoGPT-Math'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 119 (delta 29), reused 29 (delta 19), pack-reused 69 (from 1)
Receiving objects: 100% (119/119), 6.97 MiB | 3.75 MiB/s, done.
Resolving deltas: 100% (51/51), done.
/content/NanoGPT-Math


In [ ]:
import os
os.chdir("/content/NanoGPT-Math")  # make the repo root the working directory

# Install gdown if needed
!pip install gdown

# Download gpt.pt into sft/
import gdown
os.makedirs("sft", exist_ok=True)
url = "https://drive.google.com/uc?id=1gIZw-HAB-tHtEYCmNugwlIV7R3WsgjjZ"
output = "sft/gpt.pt"
gdown.download(url, output, quiet=False)

!ls sft
!ls dpo
!ls


Downloading...
From (original): https://drive.google.com/uc?id=1gIZw-HAB-tHtEYCmNugwlIV7R3WsgjjZ
From (redirected): https://drive.google.com/uc?id=1gIZw-HAB-tHtEYCmNugwlIV7R3WsgjjZ&confirm=t&uuid=f947201b-0237-4784-ae71-f6f74aad6d59
To: /content/NanoGPT-Math/sft/gpt.pt
100%|██████████| 106M/106M [00:01<00:00, 54.0MB/s]


gpt.pt	meta.pkl
dpo.ipynb  dpoTestV2.ipynb  dpoTestV3.ipynb  pos_neg_pairs.json
configurator.py  dpo  model.py	positivenegativedatapairs.py  README.md  sft


In [ ]:
import torch
torch.cuda.is_available()  # should return True
torch.cuda.get_device_name(0)  # shows GPU model


'Tesla T4'

### Step 1: Install necesscary packages

In [ ]:
!pip install matplotlib
!pip install torch numpy transformers datasets tiktoken wandb tqdm

### Step 2: Package imports and configuration

In [ ]:
import sys
import os
#sys.path.append(os.path.abspath(".."))
sys.path.append("/content/NanoGPT-Math")

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
import random
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
# Configuration
beta = 0.5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_lr = 1e-4
epochs = 5
#epochs = 10             # increased from 5
batch_size = 64
max_length = 64
num_samples = 1
max_new_tokens = 200
temperature = 0.8
#temperature = 0.6        # slightly lower to reduce garbled answers
top_k = 200
#top_k = 150              # slightly lower for more deterministic output
# tokenizer
#with open("../sft/meta.pkl", "rb") as f:
with open("sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

### Step 3: Define helper functions

In [ ]:
def compute_logprob(input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

### Step 4: Load the pretrained NanoGPT model

In [ ]:
#ckpt = torch.load("../sft/gpt.pt", map_location=device)
ckpt = torch.load("sft/gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

### Step 5: Load Data (**students are required to complete this part!**)

In [ ]:
# Load data from ./data/pos_neg_pairs.json
#with open("pos_neg_pairs.json", "r") as f:
with open("dpo/pos_neg_pairs.json", "r") as f:
    lines = json.load(f)
print(f"Loaded {len(lines)} training pairs")

# Comprehensive cleaning - only keep characters that are in stoi
print("Applying comprehensive character cleaning...")

def clean_text(text):
    # Only keep characters that exist in the tokenizer
    return ''.join(char for char in text if char in stoi)

cleaned_count = 0
for pair in lines:
    original_neg = pair['negative']
    original_pos = pair['positive']

    pair['negative'] = clean_text(pair['negative'])
    pair['positive'] = clean_text(pair['positive'])

    if original_neg != pair['negative'] or original_pos != pair['positive']:
        cleaned_count += 1

print(f"Cleaned {cleaned_count} pairs with unsupported characters")
print("Sample after comprehensive cleaning:")
print(f"Negative: {lines[0]['negative']}")
print(f"Positive: {lines[0]['positive']}")

Loaded 100000 training pairs
Applying comprehensive character cleaning...
Cleaned 100000 pairs with unsupported characters
Sample after comprehensive cleaning:
Negative: 89-39=? Sorry, I do not know
Positive: 89-39=? The answer is 50 because 89-39 equals 50.


### Step 6: Build the optimizer and scheduler (**students are required to complete this part!**)

In [ ]:
# recommend to use the AdamW optimizer
optimizer = torch.optim.AdamW(gpt.parameters(), lr=base_lr, weight_decay=0.01)
print(f"Optimizer configured with learning rate: {base_lr}")

Optimizer configured with learning rate: 0.0001


### Step 7: Begin training (**students are required to complete this part!**)

In [ ]:
# Load previous checkpoint if exists
ckpt_path = "./dpo.pt"
if os.path.exists(ckpt_path):
    print("Loading existing checkpoint...")
    checkpoint = torch.load(ckpt_path, map_location=device)
    gpt.load_state_dict(checkpoint['model_state_dict'])
    print("Checkpoint loaded. Continuing training...")
else:
    print("No checkpoint found. Starting training from scratch.")

total_steps = len(lines) // batch_size
for epoch in range(epochs):
    pbar = tqdm(get_batches(lines, batch_size))
    for step, (neg_tensor,pos_tensor) in enumerate(pbar):
        ###########################################################
        # Please complete the training code here!
        # Examples:
        # ...
        # neg_logprob
        # pos_logprob
        # loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1
        # ...
        ###########################################################
        optimizer.zero_grad()

        # Compute log probabilities
        pos_logprob = compute_logprob(pos_tensor)
        neg_logprob = compute_logprob(neg_tensor)

        # DPO loss
        loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1

        loss.backward()
        optimizer.step()

        pbar.set_description(f"Epoch {epoch+1} Loss: {loss.item():.4f}")

    ckpt_path = f"./dpo.pt"
    torch.save({
        "model_state_dict": gpt.state_dict(),
        "model_args": ckpt['model_args'],
    }, ckpt_path)
    print(f"Saved checkpoint to {ckpt_path}")

NameError: name 'os' is not defined

### Step 8: Begin testing (**students are required to complete this part!**)

In [ ]:
# Load the fine-tuned model
#ckpt_path = "../dpo/dpo.pt"
ckpt_path = "dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).cuda()
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
# Test
gpt.eval()
test_set = ["17+19=?", "3*17=?", "72/4=?", "72-x=34,x=?", "x*11=44,x=?", "3*17=?", "72/4=?", "72-x=34,x=?"]
# test_set = [
#     # Basic arithmetic (only 2-digit numbers or lower)
#     "79-7=?", "74+8=?", "1*x=6,x=?", "x+55=95,x=?",
#     "17+19=?", "3*17=?", "72/4=?", "15-8=?", "50/5=?",
#     "9*9=?", "64/8=?", "12+28=?", "6*13=?", "81/9=?",
#     "45-23=?", "7*8=?", "56/7=?", "33+67=?", "25-17=?",
#     "4*25=?", "48/12=?", "50+50=?", "75-50=?", "11*11=?",

#     # Algebra problems (2-digit numbers only)
#     "72-x=34,x=?", "x*11=44,x=?", "x+25=75,x=?", "x-15=30,x=?",
#     "x*7=63,x=?", "x/8=6,x=?", "x+18=42,x=?", "x-12=24,x=?",
#     "x*9=81,x=?", "x/6=9,x=?", "x+33=90,x=?", "x-28=15,x=?",
#     "75-x=25,x=?", "48/x=6,x=?", "15+x=40,x=?", "x-8=12,x=?",
#     "56/x=7,x=?", "27+x=50,x=?", "x-20=15,x=?", "72/x=8,x=?",
#     "36+x=60,x=?", "x-10=25,x=?", "84/x=7,x=?", "19+x=45,x=?",

#     # More edge cases (simple numbers)
#     "1+1=?", "99-98=?", "1*10=?", "50/1=?", "25/5=?",
#     "49+1=?", "50-1=?", "25*4=?", "40/4=?", "x+0=15,x=?",

#     # Division focus (all ≤ 2-digit numerators)
#     "36/6=?", "49/7=?", "81/9=?", "64/8=?", "48/12=?",
#     "64/8=?", "50/10=?", "39/13=?", "28/14=?", "30/15=?",

#     # Mixed operations (2-digit only)
#     "x*5=50,x=?", "x+40=80,x=?", "x-35=20,x=?", "x/7=7,x=?",
#     "44-x=22,x=?", "60/x=10,x=?", "x+55=90,x=?", "x-60=40,x=?"
# ]
with torch.no_grad():
    for prompt in test_set:
        prompt_ids = encode(prompt)
        ###########################################################
        # Please complete the test code here!
        # ...
        # gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        # ...
        ###########################################################
        x = torch.tensor([prompt_ids], dtype=torch.long, device=device)
        y = gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        generated_text = decode(y[0].view(-1).tolist())
        print(f"Q: {prompt}")
        print(f"A: {generated_text}")
        print("---")

# # Load the fine-tuned model
# ckpt_path = "dpo.pt"
# checkpoint = torch.load(ckpt_path, map_location=device)
# gptconf = GPTConfig(**checkpoint['model_args'])
# gpt = GPT(gptconf).cuda()
# try:
#     state_dict = checkpoint['model']
# except:
#     state_dict = checkpoint['model_state_dict']
# unwanted_prefix = '_orig_mod.'
# for k,v in list(state_dict.items()):
#     if k.startswith(unwanted_prefix):
#         state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
# gpt.load_state_dict(state_dict)

# # Test
# gpt.eval()
# test_set = [
#     # Basic arithmetic (only 2-digit numbers or lower)
#     "79-7=?", "74+8=?", "1*x=6,x=?", "x+55=95,x=?",
#     "17+19=?", "3*17=?", "72/4=?", "15-8=?", "50/5=?",
#     "9*9=?", "64/8=?", "12+28=?", "6*13=?", "81/9=?",
#     "45-23=?", "7*8=?", "56/7=?", "33+67=?", "25-17=?",
#     "4*25=?", "48/12=?", "50+50=?", "75-50=?", "11*11=?",

#     # Algebra problems (2-digit numbers only)
#     "72-x=34,x=?", "x*11=44,x=?", "x+25=75,x=?", "x-15=30,x=?",
#     "x*7=63,x=?", "x/8=6,x=?", "x+18=42,x=?", "x-12=24,x=?",
#     "x*9=81,x=?", "x/6=9,x=?", "x+33=90,x=?", "x-28=15,x=?",
#     "75-x=25,x=?", "48/x=6,x=?", "15+x=40,x=?", "x-8=12,x=?",
#     "56/x=7,x=?", "27+x=50,x=?", "x-20=15,x=?", "72/x=8,x=?",
#     "36+x=60,x=?", "x-10=25,x=?", "84/x=7,x=?", "19+x=45,x=?",

#     # More edge cases (simple numbers)
#     "1+1=?", "99-98=?", "1*10=?", "50/1=?", "25/5=?",
#     "49+1=?", "50-1=?", "25*4=?", "40/4=?", "x+0=15,x=?",

#     # Division focus (all ≤ 2-digit numerators)
#     "36/6=?", "49/7=?", "81/9=?", "64/8=?", "48/12=?",
#     "64/8=?", "50/10=?", "39/13=?", "28/14=?", "30/15=?",

#     # Mixed operations (2-digit only)
#     "x*5=50,x=?", "x+40=80,x=?", "x-35=20,x=?", "x/7=7,x=?",
#     "44-x=22,x=?", "60/x=10,x=?", "x+55=90,x=?", "x-60=40,x=?"
# ]

# # Define expected answers for automatic checking
# expected_answers = {
#     # Basic arithmetic
#     "79-7=?": "72", "74+8=?": "82", "1*x=6,x=?": "6", "x+55=95,x=?": "40",
#     "17+19=?": "36", "3*17=?": "51", "72/4=?": "18", "15-8=?": "7", "50/5=?": "10",
#     "9*9=?": "81", "64/8=?": "8", "12+28=?": "40", "6*13=?": "78", "81/9=?": "9",
#     "45-23=?": "22", "7*8=?": "56", "56/7=?": "8", "33+67=?": "100", "25-17=?": "8",
#     "4*25=?": "100", "48/12=?": "4", "50+50=?": "100", "75-50=?": "25", "11*11=?": "121",

#     # Algebra problems
#     "72-x=34,x=?": "38", "x*11=44,x=?": "4", "x+25=75,x=?": "50", "x-15=30,x=?": "45",
#     "x*7=63,x=?": "9", "x/8=6,x=?": "48", "x+18=42,x=?": "24", "x-12=24,x=?": "36",
#     "x*9=81,x=?": "9", "x/6=9,x=?": "54", "x+33=90,x=?": "57", "x-28=15,x=?": "43",
#     "75-x=25,x=?": "50", "48/x=6,x=?": "8", "15+x=40,x=?": "25", "x-8=12,x=?": "20",
#     "56/x=7,x=?": "8", "27+x=50,x=?": "23", "x-20=15,x=?": "35", "72/x=8,x=?": "9",
#     "36+x=60,x=?": "24", "x-10=25,x=?": "35", "84/x=7,x=?": "12", "19+x=45,x=?": "26",

#     # Edge cases
#     "1+1=?": "2", "99-98=?": "1", "1*10=?": "10", "50/1=?": "50", "25/5=?": "5",
#     "49+1=?": "50", "50-1=?": "49", "25*4=?": "100", "40/4=?": "10", "x+0=15,x=?": "15",

#     # Division focus
#     "36/6=?": "6", "49/7=?": "7", "81/9=?": "9", "64/8=?": "8", "48/12=?": "4",
#     "64/8=?": "8", "50/10=?": "5", "39/13=?": "3", "28/14=?": "2", "30/15=?": "2",

#     # Mixed operations
#     "x*5=50,x=?": "10", "x+40=80,x=?": "40", "x-35=20,x=?": "55", "x/7=7,x=?": "49",
#     "44-x=22,x=?": "22", "60/x=10,x=?": "6", "x+55=90,x=?": "35", "x-60=40,x=?": "100"
# }

## test_set = ["17+19=?", "3*17=?", "72/4=?", "72-x=34,x=?", "x*11=44,x=?", "3*17=?", "72/4=?", "72-x=34,x=?"]

# print(f"Testing on {len(test_set)} problems...")
# print("=" * 60)

# correct_count = 0
# total_count = len(test_set)
# results = []

# with torch.no_grad():
#     for i, prompt in enumerate(test_set):
#         prompt_ids = encode(prompt)
#         x = torch.tensor([prompt_ids], dtype=torch.long, device=device)
#         y = gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
#         generated_text = decode(y[0].view(-1).tolist())

#         # Extract just the answer part
#         answer_part = generated_text[len(prompt):].split('\n')[0].strip()

#         # Check if answer is correct
#         expected = expected_answers[prompt]
#         is_correct = expected in answer_part
#         if is_correct:
#             correct_count += 1
#             status = "✓"
#         else:
#             status = "✗"

#         results.append((prompt, answer_part, is_correct))

#         print(f"Test {i+1:2d} {status}: {prompt}")
#         print(f"        Model: {answer_part}")
#         print(f"        Expected: {expected}")
#         if not is_correct:
#             print(f"        *** WRONG ***")
#         print()

# print("=" * 60)
# print(f"FINAL RESULTS: {correct_count}/{total_count} correct")
# print(f"ACCURACY: {correct_count/total_count*100:.1f}%")
# print("=" * 60)

# # Show wrong answers summary
# print("\nWRONG ANSWERS:")
# wrong_count = 0
# for prompt, answer, correct in results:
#     if not correct:
#         wrong_count += 1
#         expected = expected_answers[prompt]
#         print(f"{wrong_count:2d}. {prompt}")
#         print(f"     Expected: {expected}")
#         print(f"     Got:      {answer}")
#         print()


Q: 17+19=?
A: 17+19=? The answer is 48 because 17+19 equals 48.
---
Q: 3*17=?
A: 3*17=? The answer is 51 because 3*17 equals 51.
---
Q: 72/4=?
A: 72/4=? The answer is 13 because 72/4 equals 13.
---
Q: 72-x=34,x=?
A: 72-x=34,x=? The answer is 46 because 72-34 equals 46.
---
Q: x*11=44,x=?
A: x*11=44,x=? The answer is 43 because 44/1 equals 4.
---
Q: 3*17=?
A: 3*17=? The answer is 51 because 3*17 equals 51.
---
Q: 72/4=?
A: 72/4=? The answer is 26 because 72/4 equals 26.
---
Q: 72-x=34,x=?
A: 72-x=34,x=? The answer is 46 because 72-34 equals 46.
---
